In [ ]:
# ==========================================================
# COMPLETE LIGHTGBM PIPELINE (SINGLE CELL)
# ==========================================================

import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib

# ----------------------------------------------------------
# LOAD DATASET
# ----------------------------------------------------------

df = pd.read_csv("Educational_Management_Dataset.csv")

print("Dataset Shape:", df.shape)

# ----------------------------------------------------------
# PREPROCESSING
# ----------------------------------------------------------

df = df.drop_duplicates()
df = df.dropna()

# ----------------------------------------------------------
# FEATURE ENGINEERING
# ----------------------------------------------------------

df["Engagement_Per_Login"] = (
    df["Video_Engagement_Time"] /
    (df["LMS_Login_Frequency"] + 1)
)

df["Performance_Index"] = (
    df["Assignment_Avg"] +
    df["Exam_Score"] +
    (df["GPA"] * 20)
) / 3

# ----------------------------------------------------------
# FEATURES & TARGET
# ----------------------------------------------------------

features = [
    "Age",
    "GPA",
    "Attendance_Rate",
    "Assignment_Avg",
    "Exam_Score",
    "LMS_Login_Frequency",
    "Video_Engagement_Time",
    "Forum_Interactions",
    "Stress_Level_Index",
    "Engagement_Per_Login",
    "Performance_Index"
]

target = "Engagement_Score"

X = df[features]
y = df[target]

# ----------------------------------------------------------
# TRAIN TEST SPLIT
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ----------------------------------------------------------
# LIGHTGBM MODEL
# ----------------------------------------------------------

model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=10,
    num_leaves=255,
    subsample=0.85,
    reg_alpha=0.05,
    reg_lambda=0.05,
    random_state=42
)

model.fit(X_train, y_train)

# ----------------------------------------------------------
# PREDICTION
# ----------------------------------------------------------

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# ----------------------------------------------------------
# METRICS
# ----------------------------------------------------------

train_mae = mean_absolute_error(y_train, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_r2 = r2_score(y_train, train_pred)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_r2 = r2_score(y_test, test_pred)

mape = np.mean(
    np.abs((y_test - test_pred) / y_test)
) * 100

# ----------------------------------------------------------
# RESULTS
# ----------------------------------------------------------

results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "MAPE (%)", "R²"],
    "Train": [
        train_mae,
        train_rmse,
        np.nan,
        train_r2
    ],
    "Test": [
        test_mae,
        test_rmse,
        mape,
        test_r2
    ]
})

print("\n================ RESULTS ================\n")
print(results)

# ----------------------------------------------------------
# FEATURE IMPORTANCE
# ----------------------------------------------------------

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Features:")
print(importance.head(10))

# ----------------------------------------------------------
# SAVE MODEL
# ----------------------------------------------------------

joblib.dump(
    model,
    "lightgbm_engagement_model.pkl"
)

print("\nModel Saved Successfully")